# Phase 1 Student Notebook: Simple Multi-Agent Orchestration

Goal: Understand a minimal multi-agent workflow using LangGraph.

## LangGraph Basics (State, Nodes, Edges)

**LangGraph** lets us build workflows as a **state machine**.

### Key Ideas
- **State** = shared memory passed between agents.
- **Node** = a function (agent) that updates state.
- **Edge** = the path between nodes.
- **Conditional Edge** = a decision (if/else) based on state.

### Flow of Information
```
User Question
    ↓
Supervisor Agent (decides route)
    ↓
Math Specialist (if math) → Final Response
    ↓
User Answer
```

Keep this in mind as we build each part below.

## 1. Imports

Run this cell first.

In [5]:
# STEP 1: Imports
# Why: We need LangGraph for orchestration and LangChain for message types.
# Hint: Load .env so OPENAI_API_KEY is available.
from dotenv import load_dotenv
load_dotenv()

from langgraph.graph import StateGraph, MessagesState
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI
from typing import Any, Literal

import json

StateGraph:
Supervisor -> Math Agent -> Final Agent

MessageState:
- Store the user messages
- Agent Replies
- Tool Outputs

## 2. LLM (Supervisor uses this)

Only the supervisor agent calls the LLM.

In [3]:
# STEP 2: Initialize the LLM
# Why: Only the supervisor uses the LLM to make routing decisions.
# Hint: temperature=0 makes it deterministic.

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 3. Math Tools

In [4]:
# STEP 3: Math Tools
# Why: Specialists call tools instead of LLMs for simple tasks.
# Hint: Keep tools pure and deterministic.
# TODO: define add_numbers(a, b)
# TODO: define multiply_numbers(a, b)
# TODO: create MATH_TOOLS dict mapping names to functions

def add_numbers(a: int, b: int) -> int:
    """Add two numbers togeather"""
    result = a + b
    print(f"Tool: {a} + {b} = {result}")
    return result

def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers togeather"""
    result = a * b
    print(f"Total: {a} x {b} = {result}")
    return result

# Tool Registry:
MATH_TOOLS = {
    "add_numbers": add_numbers,
    "multiply_numbers": multiply_numbers
}

1. Tools are a simple python functions
2. The Specialist chooses which tool to call bsed on the Supervisor decision
3. This mirrors real agents calling external tools/ API's

## 4. Supervisor Agents

### Quick Intuition: What is `state`?

Think of `state` as a plain Python dictionary that travels between agents.
Each agent reads it, adds new information, and passes it forward.

Simple example:
```python
state = {
    "messages": [HumanMessage(content="What is 2 plus 2?")]
}
messages = state.get("messages", [])  # safe read with fallback
```

- If `messages` exists, you get the list.
- If it does not, you get an empty list instead of an error.

This keeps agents simple: read from `state`, update `state`, return `state`.

In [6]:
# STEP 4: Agents
# Why: Agents are functions that read state, do work, and write state.
# Hint: Each agent must return the updated state.
# TODO: define supervisor_agent(state)
#   - read messages
#   - call llm with prompt
#   - parse JSON decision
#   - save decision + message to state
# TODO: define math_specialist_agent(state)
#   - read supervisor_decision
#   - call tool
#   - save math_result + ToolMessage to state
# TODO: define final_response_agent(state)
#   - format final_answer based on math_result
#   - save final_answer to state

def supervisor_agent(state: dict) -> dict:
    """
    Supervisor Agent: Reads the user question and decides what to do.
    Flow:
    1. Read the user's question from messages
    2. Ask LLM to analyze the question
    3. Get routing decision (math or not)
    4. Save decision to State
    """
    print("\n Supervisor: Analyzing Question")

    # Get the Conv history from state
    messages = state.get("messages", [])

    supervisor_prompt = """You are a supervisor agent managing a team of specialists.

You have access to:
- Math Specialist: Can add numbers or multiply numbers

Your job is to read the user's question and decide:
1. Is this asking to add/sum numbers? (keywords: plus, add, sum, total, +)
2. Is this asking to multiply numbers? (keywords: multiply, times, product, *)
3. If either: route to math_specialist
4. If neither: respond cannot_help

EXAMPLES:
- "What is 5 plus 3?" -> {"decision": "route_to_specialist", "specialist": "math_specialist", "operation": "add_numbers", "parameters": {"a": 5, "b": 3}}
- "Multiply 7 and 6" -> {"decision": "route_to_specialist", "specialist": "math_specialist", "operation": "multiply_numbers", "parameters": {"a": 7, "b": 6}}
- "What is the meaning of life?" -> {"decision": "cannot_help", "reason": "This is not a math question"}

IMPORTANT: Respond ONLY with the JSON object. No explanation, no markdown code blocks."""

    response = llm.invoke(messages + [SystemMessage(content = supervisor_prompt)])
    supervisor_message = response.content

    # Parse the JSON decision:
    decision = json.loads(supervisor_message) # 

    # Save to State:
    messages.append(AIMessage(content = supervisor_message, name = "supervisor"))
    state['messages'] = messages
    state['supervisor_decision'] = decision

    print(f"Decision: {decision['decision']}")

    return state

1. SystemMessage
2. HumanMessage
3. AIMessage

In [ ]:
## '{"key": "value"}' -> {"key": "value"}

## json to dict (json.loads())

### Math Specialist Agent

The Math Specialist reads the supervisor decision, calls the tools, and saves the result

- Supervisor agent -> decides
- Specialist agent -> executes

In [9]:
def math_specialist_agent(state: dict) -> dict:
    """
    Math Specialist Agent: Executes Math Operations.

    Flow:
    1. Read the supervisor's decision
    2. Get Operation (add or multiply) and parameters
    3. Call the math tool
    4. Save result to state
    """
    print("\n Math Specialist: Doing the Math prob ...")

    # Get the routing decision from supervisor:
    decision = state.get("supervisor_decision", {})
    operation = decision.get("operation")
    parameters = decision.get("parameters", {})

    print(f" Operation: {operation}")
    print(f" Paramters: {parameters}")

    tool_func = MATH_TOOLS[operation]

    result = tool_func(**parameters) ## **parameter -> unpacks dict into function arguments:

    ## Add results to messages:
    result_message = ToolMessage(
        content=f"Math operation result:{result}",
        tool_call_id=operation,
        name="math_specialist"
        )

    messages = state.get("messages", [])   
    messages.append(result_message)
    
    # Save to state:
    state['messages'] = messages
    state['math_result'] = result

    print(f"Result:{result}")
    return state

## 5. State and Routing

In [ ]:
# STEP 5: State and Routing
# Why: State is shared memory; routing decides next agent.
# Hint: Use MessagesState so messages are included.
# TODO: define OrchestratorState(MessagesState) with fields:
#   supervisor_decision, next_agent, math_result, final_answer
# TODO: define route_after_supervisor(state)
#   - if decision is route_to_specialist -> math_specialist
#   - else -> final_response

## 6. Build and Run

In [ ]:
# STEP 6: Build and Run
# Why: This wires agents into a LangGraph state machine.
# Hint: Entry point = supervisor, finish point = final_response.
# TODO: define build_orchestrator()
#   - add nodes: supervisor, math_specialist, final_response
#   - set entry point
#   - add conditional edges using route_after_supervisor
#   - add edge math_specialist -> final_response
#   - compile and return app
# TODO: define run_orchestrator(question)
#   - create initial_state with HumanMessage
#   - app.invoke(initial_state)
#   - print final answer
# TODO: test with 3 questions